---
## Exercises

### Exercise 1: Training Script
Modify `train.py` to also train a `RandomForestClassifier` and log both models' AUC to stdout as SageMaker metrics. Which performs better?

### Exercise 2: HPO Simulation
Run 25 local trials of random hyperparameter search. Plot AUC vs learning_rate. Is there a clear optimal range?

### Exercise 3: Cost Calculator
Write a function `estimate_training_cost(instance_type, duration_hours, n_trials)` that returns the estimated HPO cost. Use real SageMaker pricing.

---
## Key Takeaways

1. **Training jobs** = managed Docker containers on AWS hardware
2. **Write to SM_MODEL_DIR** — SageMaker copies your model to S3 automatically
3. **HPO** = Bayesian search over hyperparameter space (use metric_definitions!)
4. **Endpoints** = real-time, async, batch, or serverless — choose based on latency needs
5. **Always delete endpoints** when done — idle endpoints charge by the hour
6. **Local mode** = test your training script locally before spending money on SageMaker

In [ ]:
def batch_transform_simulate(model, df: pd.DataFrame) -> pd.DataFrame:
    """
    Simulate SageMaker Batch Transform.
    In production: transform job reads from S3, writes predictions to S3.
    Here: runs locally on a DataFrame.
    """
    # Feature preparation (same as training)
    feature_cols = ['tenure', 'monthly_charges', 'total']
    X = df[feature_cols] if all(c in df.columns for c in feature_cols) else df

    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    result = df.copy()
    result['churn_predicted'] = predictions
    result['churn_probability'] = probabilities.round(4)
    result['risk_tier'] = pd.cut(probabilities, bins=[0, 0.3, 0.6, 1.0],
                                  labels=['Low', 'Medium', 'High'])
    return result

print("Batch Transform pattern ready.")
print("In production: sm.create_transform_job(...)")
print("Here: batch_transform_simulate(model, df)")

---
## Section 4: Model Endpoints (Real-time Inference)

### Deploying a Trained Model
```python
# After training:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',    # $0.115/hr
    endpoint_name='churn-predictor-v1'
)

# Call the endpoint
response = predictor.predict({'instances': [{'tenure': 24, 'monthly_charges': 75}]})

# Clean up when done (IMPORTANT — endpoints cost money when idle!)
predictor.delete_endpoint()
```

### Endpoint Types
| Type | Latency | Best For |
|------|---------|---------|
| **Real-time** | < 100ms | Interactive predictions |
| **Async** | Seconds-minutes | Large payloads, batches |
| **Batch Transform** | Minutes-hours | Offline scoring of S3 files |
| **Serverless** | 0-5s cold start | Low traffic (pay per request) |

### Auto-scaling
Configure auto-scaling to handle traffic spikes:
- Scale out: > 70% CPU → add instances
- Scale in: < 20% CPU for 10 min → remove instances

In [ ]:
# HPO configuration (what you'd pass to SageMaker)
hpo_config = '''
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter

tuner = HyperparameterTuner(
    estimator=estimator,
    objective_metric_name="validation:auc",
    objective_type="Maximize",
    hyperparameter_ranges={
        "n-estimators": IntegerParameter(50, 300),
        "learning-rate": ContinuousParameter(0.01, 0.3),
        "max-depth": IntegerParameter(2, 8),
        "min-samples-leaf": IntegerParameter(5, 50),
    },
    max_jobs=20,
    max_parallel_jobs=4,
    strategy="Bayesian",
)

tuner.fit({"train": f"s3://{BUCKET}/processed/churn/train/"})
'''

print("HPO config (requires SageMaker):")
print(hpo_config)

# Simulate HPO locally with random search
print("\nSimulating HPO with local random search (10 trials):")
print("-" * 50)

results = []
for trial in range(10):
    hp = {
        'n_estimators': np.random.randint(50, 300),
        'learning_rate': np.random.uniform(0.01, 0.3),
        'max_depth': np.random.randint(2, 8),
        'min_samples_leaf': np.random.randint(5, 50),
    }
    result = simulate_sagemaker_training(hp)
    results.append(result)
    print(f"  Trial {trial+1:2d}: AUC={result['auc']:.4f}  lr={hp['learning_rate']:.3f}  depth={hp['max_depth']}")

best = max(results, key=lambda x: x['auc'])
print(f"\nBest AUC: {best['auc']:.4f}")
print(f"Best hyperparams: {best['hyperparams']}")

---
## Section 3: Hyperparameter Tuning (HPO)

SageMaker Automatic Model Tuning uses Bayesian optimization to find the best hyperparameters.

### How It Works
1. Define hyperparameter ranges (search space)
2. Set the metric to optimize (e.g., "validation:auc", maximize)
3. SageMaker runs N training jobs, uses results to guide next trials
4. Returns the best hyperparameter combination

### Search Strategies
- **Bayesian**: Learns from past trials (best for < 50 trials)
- **Random**: No learning between trials (embarrassingly parallel)
- **Grid**: Exhaustive (only for small discrete spaces)

In [ ]:
# Simulate what SageMaker training would do — runs locally
import pickle
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

def simulate_sagemaker_training(hyperparams: dict) -> dict:
    """Run the training logic locally to test before deploying to SageMaker."""
    np.random.seed(42)
    n = 5000

    # Simulate churn data
    tenure = np.random.exponential(24, n).clip(1, 72).astype(int)
    monthly_charges = np.random.normal(65, 25, n).clip(20, 150)
    churn_prob = 1 / (1 + np.exp(-(-2 + 0.05*(monthly_charges - 65) - 0.03*tenure)))
    churn = (np.random.random(n) < churn_prob).astype(int)

    X = pd.DataFrame({'tenure': tenure, 'monthly_charges': monthly_charges,
                      'total': tenure * monthly_charges})
    y = pd.Series(churn)

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    model = GradientBoostingClassifier(**hyperparams, random_state=42)
    model.fit(X_train, y_train)

    auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    return {'auc': round(auc, 4), 'hyperparams': hyperparams}

# Test with default hyperparameters
result = simulate_sagemaker_training({
    'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 10
})
print(f"Local training result: AUC = {result['auc']}")

In [ ]:
# SageMaker Estimator — configures the training job
sagemaker_code = '''
from sagemaker.sklearn.estimator import SKLearn

estimator = SKLearn(
    entry_point="train.py",           # Your training script
    source_dir="src/",                # Directory with train.py
    framework_version="0.23-1",       # scikit-learn version
    instance_type="ml.m5.large",      # $0.115/hr
    instance_count=1,
    role=SAGEMAKER_ROLE,
    hyperparameters={
        "n-estimators": 100,
        "learning-rate": 0.1,
        "max-depth": 3,
        "min-samples-leaf": 10,
    },
    metric_definitions=[
        {"Name": "validation:auc", "Regex": "validation:auc=([0-9\\.]+);"}
    ],
    enable_sagemaker_metrics=True,
)

# Launch training job
estimator.fit({
    "train": f"s3://{BUCKET}/processed/churn/train/",
})

print(f"Training job: {estimator.latest_training_job.name}")
print(f"Model artifact: {estimator.model_data}")
'''

print("SageMaker Estimator code (requires AWS account):")
print("="*60)
print(sagemaker_code)

---
## Section 2: Launching a Training Job

### The 3-Step Pattern
1. Upload training data to S3
2. Create a SageMaker Estimator (which container, which instance, which script)
3. Call `.fit()` with S3 data paths

In [ ]:
# This is the training script that runs inside SageMaker
training_script = '''#!/usr/bin/env python3
"""SageMaker training script for customer churn prediction."""

import argparse
import json
import os
import pickle
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline


def parse_args():
    parser = argparse.ArgumentParser()
    # Hyperparameters passed by SageMaker
    parser.add_argument('--n-estimators', type=int, default=100)
    parser.add_argument('--learning-rate', type=float, default=0.1)
    parser.add_argument('--max-depth', type=int, default=3)
    parser.add_argument('--min-samples-leaf', type=int, default=10)
    # SageMaker environment
    parser.add_argument('--model-dir', type=str, default=os.environ.get('SM_MODEL_DIR', './model'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAIN', './data'))
    return parser.parse_args()


def load_data(data_dir):
    """Load training data from the SageMaker input directory."""
    dfs = []
    for f in os.listdir(data_dir):
        if f.endswith('.csv'):
            dfs.append(pd.read_csv(os.path.join(data_dir, f)))
    return pd.concat(dfs, ignore_index=True)


def preprocess(df):
    """Preprocess features."""
    df = df.copy()

    # Encode categoricals
    cat_cols = ['contract_type', 'payment_method', 'internet_service']
    for col in cat_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].fillna('Unknown'))

    # Fill missing numerics
    num_cols = ['tenure_months', 'monthly_charges', 'total_charges', 'num_products', 'support_calls']
    for col in num_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    feature_cols = cat_cols + num_cols
    X = df[[c for c in feature_cols if c in df.columns]]
    y = df['churn']
    return X, y


def main():
    args = parse_args()
    os.makedirs(args.model_dir, exist_ok=True)

    print(f"Loading data from {args.train}")
    df = load_data(args.train)
    print(f"Loaded {len(df)} rows")

    X, y = preprocess(df)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    model = GradientBoostingClassifier(
        n_estimators=args.n_estimators,
        learning_rate=args.learning_rate,
        max_depth=args.max_depth,
        min_samples_leaf=args.min_samples_leaf,
        random_state=42,
    )

    print(f"Training GBM: n_est={args.n_estimators}, lr={args.learning_rate}, depth={args.max_depth}")
    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_prob)

    print(f"Validation AUC: {auc:.4f}")
    print(classification_report(y_val, y_pred))

    # SageMaker reads metrics from stdout
    print(f"validation:auc={auc:.4f};")

    # Save model
    model_path = os.path.join(args.model_dir, 'model.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump({'model': model, 'feature_cols': X.columns.tolist()}, f)
    print(f"Model saved: {model_path}")


if __name__ == '__main__':
    main()
'''

# Save to disk (this is what you'd upload to S3 and run on SageMaker)
script_path = Path('train.py')
script_path.write_text(training_script)
print(f"Training script written: {script_path}")
print("\nTo run locally for testing:")
print("  python train.py --train ./data --model-dir ./model")

---
## Section 1: Why SageMaker?

### Local Training vs SageMaker

| Aspect | Local | SageMaker |
|--------|-------|-----------|
| Hardware | Your machine | Any instance (GPU, distributed) |
| Cost | Fixed (your hardware) | Pay per job ($0.05-$3.00/hr) |
| Scalability | Limited | Up to 100s of GPUs |
| Reproducibility | Manual | Built-in (Docker containers) |
| Monitoring | You manage | CloudWatch built-in |
| Best for | Development, debugging | Production, large-scale training |

### SageMaker Training Job Anatomy
```
Your Training Script (train.py)
       ↓
Docker Container (pre-built or custom)
       ↓
SageMaker Training Job
   - Copies data from S3 to /opt/ml/input/
   - Runs your script
   - Saves artifacts to /opt/ml/model/
   - Copies model back to S3
```

### Key Environment Variables Inside Container
```python
SM_MODEL_DIR = '/opt/ml/model'      # Save your model here
SM_CHANNEL_TRAIN = '/opt/ml/input/data/train'  # Training data
SM_CHANNEL_TEST = '/opt/ml/input/data/test'    # Test data
SM_OUTPUT_DATA_DIR = '/opt/ml/output/data'     # Metrics/logs
```

In [ ]:
import boto3, json, os, time
import pandas as pd, numpy as np
from pathlib import Path
from dotenv import load_dotenv
load_dotenv('../../../.env')

AWS_REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
SAGEMAKER_ROLE = os.getenv('SAGEMAKER_ROLE_ARN', 'arn:aws:iam::123456789012:role/SageMakerRole')
BUCKET = os.getenv('S3_BUCKET', 'your-ml-bucket')

try:
    import sagemaker
    sm_session = sagemaker.Session()
    print(f"SageMaker SDK: {sagemaker.__version__}")
except ImportError:
    print("pip install sagemaker")
    print("Continuing in conceptual mode...")

print(f"Region: {AWS_REGION}")

# 02: SageMaker — Training, Tuning & Serving

## Learning Objectives
- Launch training jobs on SageMaker managed infrastructure
- Run hyperparameter tuning (HPO) with Bayesian optimization
- Deploy models as real-time endpoints
- Compare SageMaker vs local training trade-offs

## Roadmap
Training Job → Built-in Algorithms → Custom Scripts → HPO → Model Endpoints